# UMAP

## Overview

UMAP (Uniform Manifold Approximation and Projection) is a non-linear dimensionality reduction method that preserves both local and global structure better than t-SNE, runs faster, and — critically — supports `transform()` for applying to new data.

**UMAP vs t-SNE:**

| Property | t-SNE | UMAP |
|---|---|---|
| Speed | Slow (O(n log n)) | Fast |
| Global structure | Poor | Better preserved |
| New data | No `transform()` | Yes |
| Dimensions | 2–3 only | Any |
| Interpretability | None | None |

Key hyperparameters: `n_neighbors` (local vs global balance, like perplexity), `min_dist` (cluster tightness), `n_components`.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

rng = np.random.default_rng(42)
centers = [(50,7.8,1.2,0.8),(200,7.1,3.5,2.0),(350,6.5,6.0,3.5),(150,7.4,2.0,1.2)]
X_list, labels = [], []
for i, (elev,ph,nit,phos) in enumerate(centers):
    n = rng.integers(60,80)
    X_list.append(np.column_stack([
        rng.normal(elev,40,n), rng.normal(ph,0.25,n),
        rng.gamma(2,nit/2,n), rng.gamma(2,phos/2,n)
    ]))
    labels.extend([i]*n)
X = np.vstack(X_list)
labels = np.array(labels)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)
colors = ["steelblue","#e74c3c","#4fffb0","orange"]
print(f"Dataset: {X.shape}")

---
## Fitting UMAP

In [ ]:
try:
    import umap
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        random_state=42)
    X_umap = reducer.fit_transform(X_sc)
    fig, ax = plt.subplots(figsize=(7,6))
    for k in range(4):
        mask = labels == k
        ax.scatter(X_umap[mask,0], X_umap[mask,1], s=25, alpha=0.7,
                   color=colors[k], label=f"Site type {k}")
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
    ax.set_title("UMAP Embedding")
    ax.legend(); plt.tight_layout(); plt.show()
except ImportError:
    print("umap-learn not installed: pip install umap-learn")
    print("Falling back to PCA for demonstration")
    reducer = PCA(n_components=2, random_state=42)
    X_umap = reducer.fit_transform(X_sc)
    fig, ax = plt.subplots(figsize=(7,6))
    for k in range(4):
        mask = labels == k
        ax.scatter(X_umap[mask,0], X_umap[mask,1], s=25, alpha=0.7,
                   color=colors[k], label=f"Site type {k}")
    ax.set_title("PCA (umap-learn not installed)")
    ax.legend(); plt.tight_layout(); plt.show()

---
## n_neighbors and min_dist Sensitivity

In [ ]:
try:
    import umap as umap_lib
    fig, axes = plt.subplots(2,3,figsize=(13,8))
    configs = [(5,0.05),(15,0.05),(30,0.05),(15,0.01),(15,0.1),(15,0.5)]
    for ax, (nn,md) in zip(axes.flatten(), configs):
        emb = umap_lib.UMAP(n_neighbors=nn, min_dist=md,
                            random_state=42).fit_transform(X_sc)
        for k in range(4):
            ax.scatter(emb[labels==k,0], emb[labels==k,1],
                       s=10, alpha=0.7, color=colors[k])
        ax.set_title(f"n_neighbors={nn}, min_dist={md}")
        ax.set_xticks([]); ax.set_yticks([])
    plt.suptitle("UMAP Hyperparameter Sensitivity")
    plt.tight_layout(); plt.show()
except ImportError:
    print("umap-learn not installed. Install with: pip install umap-learn")
print("n_neighbors: small=local structure; large=global structure")
print("min_dist:    small=tight clusters; large=spread points")

---
## Transforming New Data (UMAP advantage over t-SNE)

In [ ]:
try:
    import umap as umap_lib
    from sklearn.model_selection import train_test_split
    X_tr, X_te, y_tr, y_te = train_test_split(X_sc, labels,
                                               test_size=0.25, random_state=42)
    reducer_fit = umap_lib.UMAP(n_components=2, n_neighbors=15,
                                random_state=42).fit(X_tr)
    X_tr_emb = reducer_fit.transform(X_tr)
    X_te_emb = reducer_fit.transform(X_te)
    fig, axes = plt.subplots(1,2,figsize=(12,5))
    for ax, emb, lbl, title in zip(axes,
        [X_tr_emb, X_te_emb], [y_tr, y_te],
        ["Training set", "Test set (transform only)"]):
        for k in range(4):
            mask = lbl == k
            ax.scatter(emb[mask,0], emb[mask,1], s=20, alpha=0.7,
                       color=colors[k], label=f"C{k}")
        ax.set_title(title); ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
    axes[0].legend()
    plt.tight_layout(); plt.show()
except ImportError:
    print("UMAP transform() allows embedding new points into the trained manifold.")
    print("t-SNE has no transform() -- every new dataset requires a full refit.")

---
## UMAP for Feature Extraction

In [ ]:
try:
    import umap as umap_lib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score
    # Compare RF on original features vs UMAP(5 components)
    reducer_5d = umap_lib.UMAP(n_components=5, n_neighbors=15, random_state=42)
    from sklearn.model_selection import train_test_split
    X_tr, X_te, y_tr, y_te = train_test_split(X_sc, labels,
                                               test_size=0.25, random_state=42)
    X_tr_u = reducer_5d.fit_transform(X_tr)
    X_te_u = reducer_5d.transform(X_te)
    rf_raw  = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)
    rf_umap = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr_u, y_tr)
    from sklearn.metrics import f1_score
    print(f"RF on original features: F1={f1_score(y_te, rf_raw.predict(X_te), average="macro"):.3f}")
    print(f"RF on UMAP(5d) features: F1={f1_score(y_te, rf_umap.predict(X_te_u), average="macro"):.3f}")
except ImportError:
    print("pip install umap-learn to use UMAP as a feature extractor for downstream models.")

---

## Common Pitfalls

**1. Treating UMAP distances as meaningful in the same way as PCA distances**  
UMAP preserves topology, not distances. Two points close in UMAP space are in the same local neighbourhood in original space, but the distance between them is not a reliable measure of dissimilarity magnitude.

**2. Using the same UMAP embedding for both exploration and model training without a train/test split**  
Fit UMAP on training data only. Fitting on all data and using the embedding as features leaks test structure into training, inflating downstream model performance.

**3. Setting n_neighbors too small (local only) or too large (global only)**  
Small `n_neighbors` (< 5) produces fragmented micro-clusters. Large values (> 50) smooth over local structure. For most tabular ecological data, 10–30 is a reasonable starting range. Always check sensitivity.

**4. Not setting random_state for reproducibility**  
UMAP is stochastic. Without `random_state`, each run produces a different layout. Always set `random_state` for any embedding you plan to report or use downstream.

**5. Using UMAP as a substitute for interpretable feature selection**  
UMAP components have no interpretable meaning — they cannot replace loadings from PCA or coefficient tables from regression. Use UMAP for visualisation and downstream modelling; use PCA loadings or permutation importance for interpretation.
---
*python_methods_library - Samantha McGarrigle*